# Banco de Dados

In [74]:
from sqlalchemy import create_engine, text
import bcrypt
import dotenv
import os
import pandas as pd

In [75]:
dotenv.load_dotenv("db_palete_caixa.env")

True

In [76]:
HOST = os.getenv("HOST")
DB_PORT = os.getenv("DB_PORT")
USER = os.getenv("USER")
DATABASE = os.getenv("DATABASE")
PASS = os.getenv("PASS")   
print(HOST, DB_PORT, USER,DATABASE)


localhost 5432 postgres db_palete_caixa


In [77]:
if PASS:
    print(f"{'\u2022' * 5}")
else:
    print("Senha não encontrada")

•••••


In [78]:
url = f"postgresql+psycopg2://{USER}:{PASS}@{HOST}:{DB_PORT}/{DATABASE}"

In [79]:
engine = create_engine(url, connect_args={"sslmode": "prefer"})

In [80]:
#import pandas as pd
df_tabelas = pd.read_sql("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_name IN ('clientes', 'tipos_status', 'tipos_usuarios')
        ORDER BY table_name;
    """, engine)
df_tabelas

,table_name


In [81]:
# --- fim 20/09/2026
# --- inicio 27/09/2026

In [82]:
# criar tabelas
cria_tabelas = """
CREATE TABLE IF NOT EXISTS tipos_status (
    id SERIAL PRIMARY KEY,
    nome_status VARCHAR(30) NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS tipos_usuarios (
    id SERIAL PRIMARY KEY,
    tipo VARCHAR(30) NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS clientes (
    id                  SERIAL PRIMARY KEY,
    cnpj                VARCHAR(14)     NOT NULL UNIQUE,
    razao_social        VARCHAR(255)    NOT NULL,
    nome_fantasia       VARCHAR(255),
    nome_contato        VARCHAR(255),
    telefone_contato    VARCHAR(20),
    email_contato       VARCHAR(255),
    nome_usuario        VARCHAR(50)     NOT NULL UNIQUE,
    senha_hash          VARCHAR(255)    NOT NULL,
    status              VARCHAR(30)     NOT NULL DEFAULT 'ativo',
    tipo                VARCHAR(30)     NOT NULL DEFAULT 'regular',
    data_criacao        TIMESTAMP       NOT NULL DEFAULT NOW(),
    deletado            BOOLEAN      NOT NULL DEFAULT FALSE,
    deletado_em         TIMESTAMP    
);

"""

with engine.begin() as conn:
    conn.execute(text(cria_tabelas))
print("Tabelas criadas com sucesso!")

Tabelas criadas com sucesso!


In [83]:
#import pandas as pd ## repliquei a celula para testar a leitura das tabelas criadas
df_tabelas = pd.read_sql("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_name IN ('clientes', 'tipos_status', 'tipos_usuarios')
        ORDER BY table_name;
    """, engine)
df_tabelas

,table_name
0,clientes
1,tipos_status
2,tipos_usuarios


In [84]:
# inserir dados nas tabelas tipos_status e tipos_usuarios
inserir_dados = """
INSERT INTO tipos_status (nome_status) VALUES
    ('ativo'),
    ('inativo'),
    ('suspenso');
   

INSERT INTO tipos_usuarios (tipo) VALUES
    ('admin'),
    ('regular');

"""

with engine.begin() as conn:
    conn.execute(text(inserir_dados))
print("Dados inseridos com sucesso!")

#     ON CONFLICT (tipo) DO NOTHING;  pode-se insluir este comando, ao final de cada bloco acima, para evitar erro de conflitos.

Dados inseridos com sucesso!


In [85]:
df = pd.read_sql(sql = "SELECT * FROM tipos_usuarios", con = engine)
df

,id,tipo
0,1,admin
1,2,regular


In [86]:
# deletar tabelas
#drop_tabelas = \
#"DROP TABLE IF EXISTS tipos_status;"
#"DROP TABLE IF EXISTS clientes;"
#"DROP TABLE IF EXISTS tipos_usuarios;"

##with engine.begin() as conn:
#    conn.execute(text(drop_tabelas))
#print("Tabelas excluidas com sucesso!")


In [87]:
# deletar tabelas
drop_tabelas = """
DROP TABLE IF EXISTS tabela_inexistente; 
--DROP TABLE IF EXISTS tipos_status;
--DROP TABLE IF EXISTS tipos_usuarios;
--DROP TABLE IF EXISTS clientes;
"""


# # truncate apaga o conteudo das tabelas, mas mantém a estrutura da tabela
# truncate_tabelas = """
# TRUNCATE TABLE IF EXISTS tabela_inexistente;  # Este é um espaço reservado para o nome real da tabela que você deseja truncar.
# TRUNCATE TABLE IF EXISTS tipos_status;
# TRUNCATE TABLE IF EXISTS clientes;
# TRUNCATE TABLE IF EXISTS tipos_usuarios;
# """


with engine.begin() as conn:
    conn.execute(text(drop_tabelas))
print("Tabelas excluidas com sucesso!")


Tabelas excluidas com sucesso!


In [88]:
# Alterar a tabela clientes para adicionar a coluna 
adicionar_coluna = """
ALTER TABLE tipos_status
ADD COLUMN IF NOT EXISTS descricao VARCHAR(255);
"""

with engine.begin() as conn:
    conn.execute(text(adicionar_coluna))
print("Coluna adicionada com sucesso!")

df_tipos_status = pd.read_sql(sql = "SELECT * FROM tipos_status", 
                              con = engine)
df_tipos_status



Coluna adicionada com sucesso!


,id,nome_status,descricao
0,1,ativo,None
1,2,inativo,None
2,3,suspenso,None


In [ ]:
# Colocar as descrições 01
update_tabela = """
UPDATE tipos_status
SET descricao = 'Usuario liberado'
WHERE nome_status = 'ativo';
"""
with engine.begin() as conn:
    conn.execute(text(update_tabela))   

df_tipos_status = pd.read_sql(sql = "SELECT * FROM tipos_status",
                              con = engine) 
df_tipos_status



,id,nome_status,descricao
0,2,inativo,NaN
1,3,suspenso,NaN
2,1,ativo,Usuario liberado


In [92]:
# Colocar as descrições 02
update_tabela = """
UPDATE tipos_status
SET descricao = 'Usuario com pendencia de pagamento'
WHERE nome_status = 'inativo';
"""
with engine.begin() as conn:
    conn.execute(text(update_tabela))   

df_tipos_status = pd.read_sql(sql = "SELECT * FROM tipos_status",
                              con = engine) 
df_tipos_status

,id,nome_status,descricao
0,3,suspenso,NaN
1,1,ativo,Usuario liberado
2,2,inativo,Usuario com pendencia de pagamento


In [93]:
# Colocar as descrições 03
update_tabela = """
UPDATE tipos_status
SET descricao = 'Usuario com licença fora do prazo'
WHERE nome_status = 'suspenso';
"""
with engine.begin() as conn:
    conn.execute(text(update_tabela))   

df_tipos_status = pd.read_sql(sql = "SELECT * FROM tipos_status",
                              con = engine) 
df_tipos_status

,id,nome_status,descricao
0,1,ativo,Usuario liberado
1,2,inativo,Usuario com pendencia de pagamento
2,3,suspenso,Usuario com licença fora do prazo


In [94]:
# 
cnpj            = "12345678901234"
razao_social    = "Prandiano"
nome_contato    = "Ricieri"
nome_usuario    = "prandiano"
tipo            = "admin" 


In [97]:
#
import getpass
senha = getpass.getpass("Insira sua senha: ")
confirmar = getpass.getpass("Confirme sua senha: ")

if senha == confirmar:
    print("Senha confirmada!")
    # Gerar o hash da senha usando bcrypt
    senha_hash = bcrypt.hashpw(senha.encode(), 
                           bcrypt.gensalt()).decode()
    print(senha_hash)
else:
    print("As senhas não coincidem. Tente novamente.")




Senha confirmada!
$2b$12$kI5FearZxoIVcY1WK4FkD.1U3V4tZr/eIHsYebvRtxvLMxfQjXuje


In [98]:
#
with engine.begin() as conn:
    # Inserir o usuário na tabela clientes
    inserir_usuario = f"""
        INSERT INTO clientes (cnpj, razao_social, nome_contato, nome_usuario, senha_hash, tipo)
        VALUES ('{cnpj}', '{razao_social}', '{nome_contato}', '{nome_usuario}', '{senha_hash}', '{tipo}')
    """
    conn.execute(text(inserir_usuario))
print("Usuário inserido com sucesso!")



Usuário inserido com sucesso!


In [99]:
#
df_clientes = pd.read_sql(sql = "SELECT * FROM clientes",
                          con = engine) 
df_clientes

,id,cnpj,razao_social,nome_fantasia,nome_contato,telefone_contato,email_contato,nome_usuario,senha_hash,status,tipo,data_criacao,deletado,deletado_em
0,1,12345678901234,Prandiano,None,Ricieri,None,None,prandiano,$2b$12$kI5FearZxoIVcY1WK4FkD.1U3V4tZr/eIHsYebv...,ativo,admin,2026-09-27 11:18:05.193578,False,None


In [100]:
# 
from sqlalchemy import Engine
from sqlalchemy import create_engine
from sqlalchemy.exc import IntegrityError

url = f"postgresql+psycopg2://{USER}:{PASS}@{HOST}:{DB_PORT}/{DATABASE}"
def get_engine() -> Engine:
    return create_engine(url, connect_args={"sslmode": "prefer"}, pool_pre_ping=True)

def disconnect_engine(engine: Engine):
    engine.dispose()

In [101]:
#
def criar_usuario(dados: dict, senha: str):
    """Insere novo usuário. Retorna None em caso de sucesso ou mensagem de erro."""
    senha_hash = bcrypt.hashpw(senha.encode(), bcrypt.gensalt()).decode()
    print("Criando usuário...")
    print(dados)
    try:
        engine = get_engine()
        with engine.begin() as conn:
            conn.execute(
                text(
                    f"""
                    INSERT INTO clientes
                        (cnpj, razao_social, nome_fantasia, nome_contato,
                         telefone_contato, email_contato, nome_usuario, senha_hash)
                    VALUES ('{dados.get('cnpj', None)}', 
                            '{dados.get('razao_social', None)}', 
                            '{dados.get('nome_fantasia', None)}', 
                            '{dados.get('nome_contato', None)}', 
                            '{dados.get('telefone_contato', None)}', 
                            '{dados.get('email_contato', None)}', 
                            '{dados.get('nome_usuario', None)}',
                            '{senha_hash}'
                            )
                    """
                )
            )
        print("Usuário criado com sucesso.")
        disconnect_engine(engine)
        return None
    except IntegrityError as e:
        diag = getattr(e.orig, "diag", None)
        constraint = (diag.constraint_name if diag else "") or ""
        if "cnpj" in constraint:
            return "CNPJ já cadastrado."
        if "nome_usuario" in constraint:
            return "Nome de usuário já em uso."
        mensagem = diag.message_primary if diag else str(e.orig)
        return f"Erro de integridade: {mensagem}"

In [102]:
#
dados = {
    'cnpj': '55123456000174',
    'razao_social': 'Empresa Exemplo LTDA',
    'nome_contato': 'João Silva',
    'telefone_contato': '11999999999',
    'nome_usuario': 'jsilva'
}
criar_usuario(dados, senha='def')

Criando usuário...
{'cnpj': '55123456000174', 'razao_social': 'Empresa Exemplo LTDA', 'nome_contato': 'João Silva', 'telefone_contato': '11999999999', 'nome_usuario': 'jsilva'}
Usuário criado com sucesso.


In [105]:
#
def autenticar(nome_usuario: str, senha: str):
    """Retorna dict do usuário se credenciais válidas e status=ativo, senão None."""
    engine = get_engine()
    with engine.connect() as conn:
        row = conn.execute(
            text(
                "SELECT id, nome_usuario, razao_social, status, tipo, senha_hash "
                f"FROM clientes WHERE nome_usuario = '{nome_usuario}' AND deletado = False"
            )).mappings().first()
    if row and bcrypt.checkpw(senha.encode(), row["senha_hash"].encode()):
        if row["status"] == "ativo":
            return dict(row)
        return {"erro": f"Acesso negado — status da conta: {row['status']}"}
    return {"erro": "Credenciais inválidas."}

autenticar(nome_usuario="prandiano", 
           senha="123")

{'id': 1,
 'nome_usuario': 'prandiano',
 'razao_social': 'Prandiano',
 'status': 'ativo',
 'tipo': 'admin',
 'senha_hash': '$2b$12$kI5FearZxoIVcY1WK4FkD.1U3V4tZr/eIHsYebvRtxvLMxfQjXuje'}

In [107]:
## para testar acima usar (auteticar):
## nome_usuario = "prandiano" senha 123
## nome_usuario = "jsilva" senha def

In [108]:
#
def listar_usuarios():
    engine = get_engine()
    with engine.connect() as conn:
        rows = conn.execute(
            text(
                "SELECT id, nome_usuario, razao_social, nome_contato, email_contato, "
                "status, tipo, data_criacao "
                "FROM clientes WHERE deletado = False ORDER BY razao_social"
            )
        ).mappings().all()
    return [dict(r) for r in rows]
listar_usuarios()

[{'id': 2,
  'nome_usuario': 'jsilva',
  'razao_social': 'Empresa Exemplo LTDA',
  'nome_contato': 'João Silva',
  'email_contato': 'None',
  'status': 'ativo',
  'tipo': 'regular',
  'data_criacao': datetime.datetime(2026, 9, 27, 11, 33, 41, 399753)},
 {'id': 1,
  'nome_usuario': 'prandiano',
  'razao_social': 'Prandiano',
  'nome_contato': 'Ricieri',
  'email_contato': None,
  'status': 'ativo',
  'tipo': 'admin',
  'data_criacao': datetime.datetime(2026, 9, 27, 11, 18, 5, 193578)}]

In [109]:
#
from datetime import datetime
def deletar_usuario(usuario_id: int):
    """Soft-delete: marca deletado=TRUE e registra a data."""
    engine = get_engine()
    with engine.begin() as conn:    # begin() abre transação e faz commit automático ao final
        conn.execute(
            text("UPDATE clientes SET deletado = TRUE, deletado_em = :agora WHERE id = :id"),
            {"agora": datetime.now(), "id": usuario_id},
        )
deletar_usuario(2)  # Exemplo de uso, substitua 1 pelo ID do usuário que deseja deletar

In [110]:
listar_usuarios()  # Verifique se o usuário foi marcado como deletado

[{'id': 1,
  'nome_usuario': 'prandiano',
  'razao_social': 'Prandiano',
  'nome_contato': 'Ricieri',
  'email_contato': None,
  'status': 'ativo',
  'tipo': 'admin',
  'data_criacao': datetime.datetime(2026, 9, 27, 11, 18, 5, 193578)}]

In [111]:
def alterar_senha(usuario_id: int, nova_senha: str):
    """Atualiza senha do usuário com nova hash."""
    nova_hash = bcrypt.hashpw(nova_senha.encode(), bcrypt.gensalt()).decode()
    engine = get_engine()
    with engine.begin() as conn:
        conn.execute(
            text(f"UPDATE clientes SET senha_hash = '{nova_hash}' WHERE id = {usuario_id}")
        )
alterar_senha(1, "def")

In [112]:
print(autenticar(nome_usuario="prandiano", senha="def"))  # Deve retornar dados do usuário
print(autenticar(nome_usuario="prandiano", senha="123"))  # Deve retornar erro de credenciais

{'id': 1, 'nome_usuario': 'prandiano', 'razao_social': 'Prandiano', 'status': 'ativo', 'tipo': 'admin', 'senha_hash': '$2b$12$088OzNcgf96G66HYVqJKYuRlU4cFc336N68Db1BHWUrnB1AT2.akq'}
{'erro': 'Credenciais inválidas.'}


In [113]:
def listar_tipos_status() -> list[str]:
    """Retorna lista de nome_status da tabela tipos_status para os dropdowns de status
    na página de alteração de status e criação de usuário."""
    engine = get_engine()
    with engine.connect() as conn:
        rows = conn.execute(text("SELECT nome_status FROM tipos_status ORDER BY nome_status")).all()
    return [r[0] for r in rows]
listar_tipos_status()

['ativo', 'inativo', 'suspenso']

In [114]:
def listar_tipos_usuario() -> list[str]:
    """Retorna lista de tipo da tabela tipos_usuarios para os dropdowns de tipo na 
    página de criação de usuário."""
    engine = get_engine()
    with engine.connect() as conn:
        rows = conn.execute(text("SELECT tipo FROM tipos_usuarios ORDER BY tipo")).all()
    return [r[0] for r in rows]
listar_tipos_usuario()

['admin', 'regular']

In [118]:
def alterar_status(usuario_id: int, novo_status: str):
    """Atualiza o status (ativo/inativo) de um usuário."""
    engine = get_engine()
    with engine.begin() as conn:    # begin() abre transação e faz commit automático ao final
        conn.execute(
            text(f"UPDATE clientes SET status = '{novo_status}' WHERE id = {usuario_id}")
        )
alterar_status(1, "inativo")

In [119]:
print(autenticar(nome_usuario="prandiano", senha="def"))  # Deve retornar dados do usuário
#print(autenticar(nome_usuario="prandiano", senha="123"))  # Deve retornar erro de credenciais

{'erro': 'Acesso negado — status da conta: inativo'}


In [120]:
alterar_status(1, "ativo")
print(autenticar(nome_usuario="prandiano", senha="def"))  # Deve retornar dados do usuário

{'id': 1, 'nome_usuario': 'prandiano', 'razao_social': 'Prandiano', 'status': 'ativo', 'tipo': 'admin', 'senha_hash': '$2b$12$088OzNcgf96G66HYVqJKYuRlU4cFc336N68Db1BHWUrnB1AT2.akq'}


In [121]:
listar_usuarios()  # Verifique se o usuário foi marcado como deletado

[{'id': 1,
  'nome_usuario': 'prandiano',
  'razao_social': 'Prandiano',
  'nome_contato': 'Ricieri',
  'email_contato': None,
  'status': 'ativo',
  'tipo': 'admin',
  'data_criacao': datetime.datetime(2026, 9, 27, 11, 18, 5, 193578)}]